# Saving ad-hoc defined configuration to BioPAP config file

In [2]:
from biopap.modules.anonymize import Anonymize
from biopap.modules.apply_ica import ApplyICA
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.baseline_correction import BaselineCorrection
from biopap.modules.bids import BIDSConversion
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.downsample import Downsample
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.load_data import LoadData
from biopap.modules.plot import Plot, PlotTypes
from biopap.modules.rereference import Rereference
from biopap.modules.segmentation import Segmentation
from biopap.modules.save_output import SaveOutput
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage

from biopap.modules.helpers import save_modules_config

In [3]:
# have a data file and load data
data_file = '/my/data/ABC1234.bdf'
load = LoadData(file_path=data_file)

Extracting EDF parameters from /data/BioPAP/data/CBN1BD071212-Cropped.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 3276799  =      0.000 ...  1600.000 secs...


In [4]:
# Create the modules you want to use, adding and testing the configration for each until 
# you are satisfied that each module's settings are correct.

ano = Anonymize(
    raw=load.raw,
    daysback=100,
    experimenter='J. Jansen',
    participant_id='AB00123')

In [10]:
# You can use both the standard variables - in this case `ref_channels` -, as well as add
# additional parameters to be passed on the the underlying mne-function, like `ch_type` in the example below.

ref = Rereference(
    raw=ano.processed,
    ref_channels=['EXG1', 'EXG2'],
    ch_type='auto')

EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.


In [11]:
plt = Plot(
    raw=ano.processed,
    save_folder='./',
    save_postfix='IRR')

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
wrote plot to 'AB00123_plot_IRR_(1).png'


In [12]:
eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])

EEG channel type selected for re-referencing


Error in sys.excepthook:
Traceback (most recent call last):
  File "/home/maarten/anaconda3/envs/biopap/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 2205, in showtraceback
    self._showtraceback(etype, value, stb)
  File "/home/maarten/anaconda3/envs/biopap/lib/python3.11/site-packages/ipykernel/zmqshell.py", line 656, in _showtraceback
    dh.parent_header,  # type:ignore[attr-defined]
    ^^^^^^^^^^^^^^^^
  File "/home/maarten/anaconda3/envs/biopap/lib/python3.11/site-packages/ipykernel/displayhook.py", line 86, in parent_header
    return self._parent_header.get()
           ^^^^^^^^^^^^^^^^^^^^^^^^^
LookupError: <ContextVar name='parent_header' at 0x709ea74b9440>

Original exception was:
Traceback (most recent call last):
  File "/home/maarten/anaconda3/envs/biopap/lib/python3.11/site-packages/mne_qt_browser/_pg_figure.py", line 176, in run
    data = np.concatenate((data, data_chunk), axis=1)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError

Creating RawArray with float64 data, n_channels=2, n_times=3276800
    Range : 0 ... 3276799 =      0.000 ...  1600.000 secs
Ready.
Added the following bipolar channels:
hEOG, vEOG


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/eog_channels.py:77: RuntimeWarning: The unit for channel(s) hEOG, vEOG has changed from NA to V.
  self.processed.set_channel_types({eog_ch: 'eog' for eog_ch in ref_channels})


In [13]:
channel_types = {
    'misc': ['EXG1','EXG2','EXG3','EXG4','EXG5','EXG6','EXG7','EXG8','Erg1','Erg2','Plet'],
    'gsr': ['GSR1', 'GSR2'],
    'resp': ['Resp'],
    'temperature': ['Temp']
}

montage = SetChannelTypeAndMontage(
    raw=eog.processed,
    channel_types=channel_types,
    mne_standard_montage='biosemi64'
)

In [14]:
flt = IIRFilter(
    raw=montage.processed,
    l_freq=0.1,
    h_freq=35)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.10, 35.00 Hz: -6.02, -6.02 dB



In [15]:
dwn = Downsample(
    raw=flt.processed,
    sfreq=500)  # Downsample to 500 Hz

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  2

/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/downsample.py:52: RuntimeWarning: Resampling of the stim channels caused event information to become unreliable. Consider finding events on the original data and passing the event matrix as a parameter.
  self.processed.resample(** {


In [16]:
blink = BlinkDetection(raw=dwn.processed)

Using EOG channels: hEOG, vEOG
EOG channel index for this subject is: [80 81]
Filtering the data to remove DC offset to help distinguish blinks from saccades
Selecting channel vEOG for blink detection
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5000 samples (10.000 s)

Now detecting blinks and generating corresponding events
Found 235 significant peaks
Number of EOG events detected: 235


In [17]:
# Make sure when you have multiple instances of the same module in your pipeline
# to give each instance a different name.

plt2 = Plot(
    raw=blink.processed,
    title="Show blink annotations")

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Channels marked as bad:
none


In [18]:
seg = Segmentation(
    raw=blink.processed,  
    stim_channel='Status',  
    start=-0.2,  
    end=0.8,     
    baseline=(None, 0),
    reference_markers={'range': '1-69'})

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Not setting metadata
540 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 540 events and 501 original time points ...
20 bad epochs dropped


In [19]:
artifact = ArtifactReject(
    raw=seg.processed,
    mode='interpolation',
    max_gradient_uv=50,
    max_amplitude_uv=75,
    min_activity_uv=0.5,
    interval_ms=100)

Fp1: bad in 3 epochs
AF7: bad in 2 epochs
AF3: bad in 1 epochs
F1: bad in 0 epochs
F3: bad in 0 epochs
F5: bad in 0 epochs
F7: bad in 0 epochs
FT7: bad in 0 epochs
FC5: bad in 0 epochs
FC3: bad in 0 epochs
FC1: bad in 0 epochs
C1: bad in 0 epochs
C3: bad in 0 epochs
C5: bad in 0 epochs
T7: bad in 0 epochs
TP7: bad in 0 epochs
CP5: bad in 0 epochs
CP3: bad in 0 epochs
CP1: bad in 0 epochs
P1: bad in 0 epochs
P3: bad in 0 epochs
P5: bad in 0 epochs
P7: bad in 0 epochs
P9: bad in 0 epochs
PO7: bad in 0 epochs
PO3: bad in 0 epochs
O1: bad in 1 epochs
Iz: bad in 0 epochs
Oz: bad in 0 epochs
POz: bad in 0 epochs
Pz: bad in 3 epochs
CPz: bad in 0 epochs
Fpz: bad in 3 epochs
Fp2: bad in 3 epochs
AF8: bad in 1 epochs
AF4: bad in 0 epochs
AFz: bad in 0 epochs
Fz: bad in 0 epochs
F2: bad in 0 epochs
F4: bad in 0 epochs
F6: bad in 0 epochs
F8: bad in 0 epochs
FT8: bad in 0 epochs
FC6: bad in 0 epochs
FC4: bad in 0 epochs
FC2: bad in 0 epochs
FCz: bad in 0 epochs
Cz: bad in 0 epochs
C2: bad in 0 ep

In [20]:
baseline = BaselineCorrection(
    raw=artifact.processed,
    baseline=(-0.2, 0))

Applying baseline correction (mode: mean)


In [21]:
ica_result = ApplyICA(
    raw=baseline.processed,
    n_components=None, # use all components
    method='fastica',
    silent=False)

Fitting ICA to data using 64 channels (please be patient, this may take a while)


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/apply_ica.py:51: RuntimeWarning: The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.
  ica = mne.preprocessing.ICA(n_components=n_components, method=method, **i_kwargs).fit(self.processed, **f_kwargs)


Selecting by non-zero PCA components: 64 components
Fitting ICA took 20.5s.
Applying ICA to Epochs instance
    Transforming to ICA space (64 components)
    Zeroing out 0 ICA components
    Projecting back using 64 PCA components


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/apply_ica.py:52: RuntimeWarning: The data you passed to ICA.apply() was baseline-corrected. Please note that ICA can introduce DC shifts, therefore you may wish to consider baseline-correcting the cleaned data again.
  self.processed = ica.apply(self.processed, **a_kwargs)


In [22]:
plt3 = Plot(
    raw=ica_result.processed,
    title="Preprocessed data")

You seem to have overlapping epochs. Some event lines may be duplicated in the plot.


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/plot.py:174: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  _ = self.processed.plot(** args | { 'block': True, 'show': True } | kwargs)


Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


In [23]:
save_output = SaveOutput(
    raw=ica_result.processed,
    export_formats='fif',
    output_file='./processed_data-epo.fif',
    overwrite=True)

wrote 'processed_data-epo__partial.fif' (85,479,520)


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/save_output.py:109: RuntimeWarning: This filename (processed_data-epo__partial.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  self.processed.save(f_new, overwrite=overwrite)


Once you are done with adding modules and editing their configuration, the settings for the complete pipeline can be saved to a configration file by calling the `save_modules_config` helper function.

Note that you only need to pass the function a list of the modules that you actually want added to the config. For instance, the example below omits `plt2` and `plt3`, the second plotting step. Additionally, it is not necessary to save the settings from the Load module, as that module is also integrated in the Pipeline class.

In [24]:
cfg_file = './config.yaml'
save_modules_config(
    modules=[ano, ref, plt, eog, flt, dwn, seg, artifact, baseline, ica_result, save_output], 
    config_path=cfg_file
)